# NLP Job Screener — Preprocessing

In [1]:
import importlib
import sys

import numpy as np
import pandas as pd
from pathlib import Path

sys.path.append('../')

import src.utils.text_utils as text_utils
importlib.reload(text_utils)
from src.utils.text_utils import clean_text

In [2]:
BASE = Path('../data/raw')

df_resume = pd.read_csv(BASE / 'resume-dataset/Resume/Resume.csv')
df_jobs = pd.read_csv(BASE / 'linkedin-job-postings/postings.csv')

In [3]:
df_resume.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
df_resume = df_resume.drop(columns=['ID', 'Resume_html'])

In [5]:
df_resume = df_resume[df_resume['Resume_str'].str.split().str.len() != 0]
#df_resume = df_resume[df_resume['Resume_str'].str.strip() != '']

In [6]:
sample_before = df_resume['Resume_str'].iloc[0]

df_resume['Resume_str'] = df_resume['Resume_str'].apply(clean_text)

sample_after = df_resume['Resume_str'].iloc[0]

print("BEFORE:\n", sample_before[:300])
print("\nAFTER:\n", sample_after[:300])

BEFORE:
          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commit

AFTER:
 hr administrator marketing associate hr administrator summary dedicated customer service manager with 15 years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service h


In [7]:
df_resume.to_csv('../data/processed/resume_clean.csv', index=False)

In [8]:
df_jobs.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [9]:
df_jobs =df_jobs[['job_id', 'title', 'description', 'formatted_experience_level', 'location', 'formatted_work_type']]

In [10]:
df_jobs = df_jobs[(df_jobs['description'].notna()) & (df_jobs['description'].str.split().str.len() >= 50)]
df_jobs['description'] = df_jobs['description'].apply(clean_text)

In [11]:
print(df_jobs.shape)

(122579, 6)


In [12]:
df_jobs.to_parquet("../data/processed/jobs_clean.parquet")


In [21]:
import re
import importlib
import src.utils.category_mapping as category_mapping

importlib.reload(category_mapping)
from src.utils.category_mapping import CATEGORY_KEYWORDS

def predict_category(title):
    if not isinstance(title, str):
        return "OTHER"

    title_clean = f" {title.lower()} "
    title_clean = re.sub(r'[^\w\s]', ' ', title_clean)
    title_clean = re.sub(r'\s+', ' ', title_clean)

    scores = {category: 0 for category in CATEGORY_KEYWORDS.keys()}

    for category, keywords in CATEGORY_KEYWORDS.items():
        for keyword in keywords:
            if keyword.lower() in title_clean:
                scores[category] += len(keyword.split())

    best_category = max(scores, key=scores.get)
    return best_category if scores[best_category] > 0 else "OTHER"


df_jobs['category'] = df_jobs['title'].apply(predict_category)

In [22]:
df_jobs['category'].value_counts()

category
INFORMATION-TECHNOLOGY    31602
MANAGEMENT                25692
OTHER                     19023
HEALTHCARE                10571
ENGINEERING                8899
SALES                      6333
FINANCE                    4363
PUBLIC-RELATIONS           3272
BPO                        2157
ACCOUNTANT                 1942
ADVOCATE                   1274
BANKING                    1158
HR                         1130
DESIGNER                   1002
CONSULTANT                  858
TEACHER                     773
DIGITAL-MEDIA               686
AUTOMOBILE                  560
BUSINESS-DEVELOPMENT        445
CONSTRUCTION                304
CHEF                        156
AVIATION                    119
ARTS                        102
FITNESS                     100
AGRICULTURE                  34
APPAREL                      24
Name: count, dtype: int64

In [23]:
import pandas as pd
import random

def build_matching_pairs(df_resume, df_jobs, n_positive=3):
    pairs = []
    n_negative = n_positive * 3

    jobs_by_category = {cat: group for cat, group in df_jobs.groupby('category')}
    all_categories = list(jobs_by_category.keys())

    for _, resume in df_resume.iterrows():
        category = resume['Category']
        resume_text = resume['Resume_str']

        if category in jobs_by_category:
            pos_pool = jobs_by_category[category]
            pos_samples = pos_pool.sample(n=min(len(pos_pool), n_positive))

            for _, job in pos_samples.iterrows():
                pairs.append({
                    'resume_text': resume_text,
                    'job_title': job['title'],
                    'job_description': job['description'],
                    'label': 1
                })

        other_categories = [c for c in all_categories if c != category]
        if other_categories:
            neg_pool = pd.concat([jobs_by_category[c] for c in other_categories])

            if not neg_pool.empty:
                neg_samples = neg_pool.sample(n=min(len(neg_pool), n_negative))

                for _, job in neg_samples.iterrows():
                    pairs.append({
                        'resume_text': resume_text,
                        'job_title': job['title'],
                        'job_description': job['description'],
                        'label': 0
                    })

    return pd.DataFrame(pairs)

In [24]:
df_pairs = build_matching_pairs(df_resume, df_jobs, n_positive=3)
print(df_pairs.shape)
print(df_pairs['label'].value_counts())

(29796, 4)
label
0    22347
1     7449
Name: count, dtype: int64


In [25]:
df_pairs

,resume_text,job_title,job_description,label
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1
3,hr administrator marketing associate hr admini...,Quality Supervisor,location sumter sc about skf skf has been maki...,0
4,hr administrator marketing associate hr admini...,Corporate Director of Reimbursement (Exempt),facility cape fear valley medical center locat...,0
...,...,...,...,...
29791,storekeeper ii professional summary the purpos...,Executive Assistant,want to learn more about this role and jobot c...,0
29792,storekeeper ii professional summary the purpos...,"BST Engagement Lead, AppSec, AppSec - Amazon S...",description in amazon stores we ship some of t...,0
29793,storekeeper ii professional summary the purpos...,Investment Operations Manager,positionposition title investment operations m...,0
29794,storekeeper ii professional summary the purpos...,Environmental & Waste Water Treatment Manager,the ideal candidate will be responsible for ma...,0


In [26]:
df_pairs.to_csv('../data/processed/matching_pairs.csv', index=False)

## Preprocessing Summary

### Resume Dataset
- Removed 1 empty resume (`Resume_str` length == 0)
- Dropped columns: `ID`, `Resume_html`
- Applied `clean_text()`: HTML tags removed, lowercased, special characters replaced with spaces, whitespace normalized
- Saved to: `data/processed/resume_clean.csv`

### Jobs Dataset
- Selected columns: `job_id`, `title`, `description`, `formatted_experience_level`, `location`, `formatted_work_type`
- Removed 7 rows with missing `description`
- Removed 1,263 job descriptions shorter than 50 words
- Applied `clean_text()` to `description`
- Mapped job titles to categories using `CATEGORY_KEYWORDS` dictionary (`predict_category()`)
- Saved to: `data/processed/jobs_clean.parquet`

### Matching Pairs
- Built using synthetic negative sampling (1:3 positive/negative ratio)
- Positive pairs (label=1): resume category matches job category
- Negative pairs (label=0): resume category does not match job category
- Total pairs: 29,796 (7,449 positive / 22,347 negative)
- Saved to: `data/processed/matching_pairs.csv`

### Reusable Utilities
- `src/utils/text_utils.py` — `clean_text()` function
- `src/utils/category_mapping.py` — `CATEGORY_KEYWORDS` dictionary + `predict_category()` function